# Week 9: Semantic Search and Retrieval Augmented Generation (RAG)

In Week 8, Sara chatted with the AeroBeat support bot, and we kept her earlier messages by pasting them back into the prompt. But ask that bot about AeroBeat's return policy and it has nothing to paste. It never saw AeroBeat's help center during training, so it guesses.

This week we give the model the right pages to read before it answers.

| Class | Part | Question it answers |
|---|---|---|
| 1 | **A. Semantic search** | Given a question, which help-center text is relevant? |
| 2 | **B. Retrieval Augmented Generation** | How do we make the model answer from that text? |

The idea carries straight over from Week 8: **memory pasted old turns into the prompt; RAG pastes retrieved documents.**

---
# A. Semantic Search

## Setup

We embed text with the same model as Week 6, `all-MiniLM-L6-v2`: a small sentence encoder that turns any text into a 384-number vector.

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(embedder.encode("not bad at all, actually pretty good").shape)

d:\Documents\Razi\Learning\LLM\CPS 535\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8076.83it/s]


(384,)


## The knowledge base

AeroBeat's help center: 14 short articles plus one long user guide. Each entry is a `(title, text)` pair. Two details are deliberately unusual so the model cannot guess them later: returns are allowed for **45 days**, and the factory reset needs **both volume buttons for 15 seconds**.

In [2]:
DOCS = [
    ("Returns and refunds", "You can return any AeroBeat product within 45 days of delivery for a full refund. Start a return from the Orders page and print the prepaid label. Products must include the original box and all accessories. Refunds go to your original payment method within 7 business days of the item reaching our warehouse."),
    ("Shipping and delivery", "We send every order with tracking. Standard shipping takes 3 to 5 business days and is free on orders over $50. If you would like faster delivery, Express shipping takes 1 to 2 business days and costs $12."),
    ("Warranty", "Every AeroBeat product comes with a 2-year limited warranty that covers manufacturing defects, such as a dead speaker or a cracked hinge. We repair or replace the product at no cost. The warranty does not cover water damage, lost parts, or damage from drops."),
    ("Battery and charging", "The AeroBeat Pro 2 plays music for up to 40 hours on a full charge with noise cancelling off, or 30 hours with it on. A full charge takes about 2 hours with the included USB-C cable. Ten minutes of charging gives about 4 hours of playback."),
    ("AeroBeat Buds charging case", "The AeroBeat Buds earbuds play for 6 hours, and the charging case holds 3 extra full charges, for 24 hours in total. The case charges over USB-C or on any wireless charging pad."),
    ("Pairing with Bluetooth devices", "To pair, hold the power button for 5 seconds until the light flashes blue, then choose AeroBeat Pro 2 in the Bluetooth settings of your phone, tablet, or computer. The headphones remember up to 8 devices and reconnect to the last one automatically."),
    ("Noise cancelling modes", "Press the ANC button once to cycle through three modes: noise cancelling, transparency, and off. Noise cancelling reduces steady sounds such as engines and air conditioning. Transparency mode lets outside sound in so you can hear announcements and conversations."),
    ("Water and sweat resistance", "The AeroBeat Buds are rated IPX4, so they handle sweat and light rain. The AeroBeat Pro 2 is not water resistant and should be kept away from rain, showers, and pools."),
    ("Changing or cancelling an order", "You can change the color or cancel an order within 2 hours of placing it from the Orders page. After 2 hours the order is packed and can no longer be changed, but you can still return it once it arrives."),
    ("Payment methods", "We accept Visa, Mastercard, American Express, PayPal, and Apple Pay. Purchases over $100 can be split into four interest-free installments."),
    ("Cleaning and ear cushions", "Wipe the ear cushions and headband with a soft, slightly damp cloth. Do not use alcohol or soap. Replacement ear cushions cost $19 a pair and snap on without tools."),
    ("Price match", "If you find the same AeroBeat product for less at an authorized retailer within 14 days of your purchase, contact us and we will refund the difference."),
    ("Contact customer support", "Our support team is available by live chat and phone, Monday to Friday, 9 am to 6 pm Eastern Time. You can also email support@aerobeat.example and we reply within one business day."),
    ("AeroBeat app", "The free AeroBeat app for iOS and Android lets you adjust the equalizer, choose noise cancelling levels, and install firmware updates. Turn on Find My Headphones in the app to see where your headphones were last connected."),
    ("AeroBeat Pro 2 user guide", "\n\n".join([
        "In the box. Your AeroBeat Pro 2 comes with a carrying case, a USB-C charging cable, a 3.5 mm audio cable for wired listening, and an airplane adapter. Keep the box and accessories in case you need to return or exchange the headphones.",
        "Controls. The right earcup has three buttons: power, volume up, and volume down. The left earcup has the ANC button. Tap the outside of the right earcup once to play or pause, twice to skip forward, and three times to go back. Swipe up or down on the earcup to change the volume without pressing the buttons.",
        "Wearing and fit. Extend the headband until the cushions sit fully around your ears, not on top of them. A good seal improves bass and makes noise cancelling work better. If the headphones press too hard, store them flat for a day to relax the headband.",
        "Wired listening. Plug the 3.5 mm cable into the left earcup to listen without Bluetooth, for example on airplane entertainment systems. Wired listening works even when the battery is empty, but noise cancelling and touch controls need power.",
        "Multipoint. The headphones can stay connected to two devices at the same time, such as a laptop and a phone. Music plays from whichever device started it last, and an incoming call on either device takes priority.",
        "Auto pause. A sensor inside the right earcup pauses music when you take the headphones off and resumes it when you put them back on. You can turn this off in the AeroBeat app.",
        "Sleep mode. If the headphones are not connected for 30 minutes, they turn off to save battery. Fold the earcups flat to put them into deep sleep for travel.",
        "Factory reset. If the headphones freeze or will not connect to any device, do a factory reset: hold the volume up and volume down buttons together for 15 seconds until the light blinks red three times. This erases all paired devices, so you will need to pair your phone again.",
        "Safety. Listening at high volume for long periods can damage your hearing. Keep the volume at a comfortable level and take breaks.",
    ])),
]
print(len(DOCS), "documents")

15 documents


## Naive version: keyword search

The classic approach scores documents by shared words. TF-IDF gives each word a weight: high if it is frequent in this document but rare across the collection. A query and a document are compared by the cosine of their TF-IDF vectors, so a document can only score above zero if it **shares a word** with the query. Common words like "can", "them", and "back" are dropped as stop words.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

doc_titles = [t for t, _ in DOCS]
doc_texts  = [f"{t}. {x}" for t, x in DOCS]          # title + text, so titles are searchable too

tfidf = TfidfVectorizer(stop_words="english")
doc_tfidf = tfidf.fit_transform(doc_texts)            # (15 docs, vocabulary size)

def keyword_search(query, k=3):
    scores = (doc_tfidf @ tfidf.transform([query]).T).toarray().ravel()
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), doc_titles[i], doc_texts[i]) for i in top]

def show(hits):
    for score, title, _ in hits:
        print(f"  {score:6.3f}  {title}")

query = "Can I send them back if I don't like them?"
print("Words that survive:", tfidf.build_analyzer()(query))
show(keyword_search(query))

Words that survive: ['send', 'don', 'like']
   0.258  Shipping and delivery
   0.000  Returns and refunds
   0.000  Warranty


The top hit is **Shipping and delivery**, because it contains "send" ("We send every order") and "like" ("If you would like faster delivery"). The article Sara needs, Returns and refunds, scores **0**: it says "return" and "refund", never "send back". Keyword search matches spelling, not meaning.

## The fix: Search by meaning

Semantic search replaces word counts with embeddings, where texts with similar meaning sit close together. We take three of our 15 articles and Sara's question and do the whole search by hand.

**The vectors are made up for the board.** Real MiniLM vectors have 384 dimensions learned in training, and no single dimension has a clean meaning. Here we pretend there are two: dimension 1 is "returning or refunding something", dimension 2 is "shipping or delivering something".

| Text | Main meaning | Embedding |
|---|---|---|
| Query: "Can I send them back if I don't like them?" | return a product | $q = [0.9,\ 0.1]$ |
| D1: Returns and refunds | return a product, get a refund | $[1.0,\ 0.0]$ |
| D2: Shipping and delivery | send an order, track it | $[0.2,\ 0.98]$ |
| D3: Warranty | repair a defective product | $[0.2,\ 0.4]$ |

The query contains "send", but its meaning sits near D1, not D2.

### Step 1: Cosine similarity

$$\operatorname{sim}(A, B) = \frac{a_1 b_1 + a_2 b_2}{\sqrt{a_1^2 + a_2^2}\ \sqrt{b_1^2 + b_2^2}}$$

The numerator is the dot product; the denominator divides out the lengths of the two vectors.

Query length, used in every calculation: $\lVert q \rVert = \sqrt{0.9^2 + 0.1^2} = \sqrt{0.81 + 0.01} = \sqrt{0.82} \approx 0.906$

**A. Query vs. D1 (Returns).** Dot product: $(0.9)(1.0) + (0.1)(0.0) = 0.9$. Length: $\lVert D_1 \rVert = \sqrt{1^2 + 0^2} = 1$.
$$\operatorname{sim}(q, D_1) = \frac{0.9}{0.906 \times 1} \approx 0.994$$

**B. Query vs. D2 (Shipping).** Dot product: $(0.9)(0.2) + (0.1)(0.98) = 0.18 + 0.098 = 0.278$. Length: $\lVert D_2 \rVert = \sqrt{0.04 + 0.9604} = \sqrt{1.0004} \approx 1.000$.
$$\operatorname{sim}(q, D_2) = \frac{0.278}{0.906 \times 1.000} \approx 0.307$$

**C. Query vs. D3 (Warranty).** Dot product: $(0.9)(0.2) + (0.1)(0.4) = 0.18 + 0.04 = 0.22$. Length: $\lVert D_3 \rVert = \sqrt{0.04 + 0.16} = \sqrt{0.20} \approx 0.447$.
$$\operatorname{sim}(q, D_3) = \frac{0.22}{0.906 \times 0.447} \approx 0.543$$

Ranking: **D1 Returns (0.994)**, D3 Warranty (0.543), D2 Shipping (0.307).

### Step 2: Compare with keyword search

| Method | Top document | Why |
|---|---|---|
| TF-IDF (A.2) | Shipping and delivery | It contains the exact word "send" |
| Semantic search | Returns and refunds | Its embedding points in nearly the same direction as the query |

TF-IDF compares matching words; semantic search compares the direction of vectors that represent meaning. A real model learns those directions in training, so its scores will differ from our invented ones. A.4 runs the real thing.

## Semantic search with MiniLM

Let's work with real embeddings. One shortcut: `normalize_embeddings=True` scales every vector to length 1, so the cosine denominator becomes $1 \times 1$ and cosine is just the dot product. Stacking the document vectors as rows of a matrix then scores every document in one line, `index["vecs"] @ q_hat`. `build_index()` and `search()` are reused for the rest of the week.

In [4]:
def build_index(pieces):                              # pieces: list of (title, text)
    texts = [f"{t}. {x}" for t, x in pieces]
    vecs = embedder.encode(texts, normalize_embeddings=True)          # unit vectors, one row per text: (n, 384)
    return {"titles": [t for t, _ in pieces], "texts": texts, "vecs": vecs}

def search(query, index, k=3):
    q_hat = embedder.encode(query, normalize_embeddings=True)         # unit vector: (384,)
    scores = index["vecs"] @ q_hat                                    # cosine with every document at once
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), index["titles"][i], index["texts"][i]) for i in top]

doc_index = build_index(DOCS)
print(doc_index["vecs"].shape)
show(search(query, doc_index))

(15, 384)
   0.308  Returns and refunds
   0.303  Changing or cancelling an order
   0.228  Contact customer support


Returns and refunds is now first, even though it shares no word with the question (TF-IDF gave it 0). The runner-up, Changing or cancelling an order, is close behind for a good reason: it ends with "you can still return it once it arrives." Don't read the scores as percentages; with MiniLM, a question and a paragraph rarely score above 0.6, and only the ranking matters. A gap this small is why RAG passes the top 3 to the model rather than only the top 1.

**This index is a vector database.** A vector database stores vectors with their text and returns the nearest ones to a query, which is exactly `doc_index` plus `search()`. Scoring every vector (exact search) is fine for thousands of documents. For millions, libraries such as FAISS use approximate search: group similar vectors ahead of time and score only the groups near the query, trading a little accuracy for a large speedup.


---
# B. Retrieval Augmented Generation
Make sure to run all the cells above before proceeding.

## The generator from Week 8

Same `Qwen2.5-1.5B-Instruct`, same greedy settings, same helpers. Brought forward unchanged from Week 8.

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()
model.generation_config.do_sample = False          # greedy: same output for everyone
model.generation_config.temperature = model.generation_config.top_p = model.generation_config.top_k = None

def to_prompt(messages):                     # the chat template text from Week 7
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate(messages, max_new_tokens=100):
    ids = tokenizer(to_prompt(messages), return_tensors="pt", add_special_tokens=False).to(device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def count_tokens(messages):
    return len(tokenizer(to_prompt(messages), add_special_tokens=False)["input_ids"])

SUPPORT = "You are a customer support assistant for AeroBeat, an online headphone shop."

Loading weights: 100%|██████████| 338/338 [00:05<00:00, 63.22it/s] 


## Naive version: ask without retrieval

Two questions whose answers exist only in AeroBeat's help center.

In [6]:
QUESTIONS = ["How many days do I have to return my AeroBeat headphones?",
             "My headphones are frozen. How do I reset them to factory settings?"]

for q in QUESTIONS:
    print("Q:", q)
    print("A:", generate([{"role": "system", "content": SUPPORT}, {"role": "user", "content": q}]), "\n")

Q: How many days do I have to return my AeroBeat headphones?
A: At AeroBeat, we strive to provide our customers with the best possible experience and satisfaction. We understand that sometimes things don't go as planned, which is why we offer a hassle-free 30-day return policy on all products.

Here's how it works:

1. **Return Period**: You can return your headphones within 30 days of receiving them.
2. **Reasons for Return**: The product must be in its original condition (unopened, unused, and not worn) and 

Q: My headphones are frozen. How do I reset them to factory settings?
A: I'm sorry to hear that your headphones are experiencing issues. To help you reset them to their factory settings, follow these steps:

1. **Unplug the Headphones**: Ensure that all power cords and cables are disconnected from both the device and the headphones.

2. **Power Off Completely**: Turn off the device where the headphones are connected (e.g., computer or mobile phone).

3. **Remove Batteries** (if 

The answers sound confident and are made up. The model has never seen AeroBeat's policies, so it produces what is typical for shops in general: a "hassle-free 30-day return policy" (the real one is 45 days) and reset steps such as unplugging the cables and removing the batteries. This is the same failure as Week 8's "John": fluent text with nothing behind it.

The model is not broken; it is missing information. Part A already built something that finds that information.

## What is RAG?

**Retrieval Augmented Generation (RAG)** is a way of answering questions in which a language model is given relevant text from an outside collection of documents at the moment it answers. A retriever finds the text, and it is placed in the prompt so the answer is grounded in those documents rather than only in what the model memorized during training.

The name lists the three steps (slides 12 to 14):

1. **Retrieve:** search the document collection for the chunks most relevant to the question. This is Part A's `search()`.
2. **Augment:** add those chunks to the prompt, next to the question.
3. **Generate:** the language model writes the answer from the augmented prompt. This is Week 8's `generate()`.

Three things follow from this design:

- **No weights change.** Fine-tuning teaches a model by training it; RAG teaches it by showing it the right page.
- **Knowledge is easy to update.** Edit a document and the next answer reflects it.
- **Answers can be checked.** Each answer can cite the source it came from.

Why does pasting text in work at all? The model's attention can look at every token in its context, so an answer that appears in the prompt can be read from there. RAG is the same idea as Week 8's memory: **memory pasted old turns into the prompt; RAG pastes retrieved documents.**

## Preparing the documents: chunking

Whatever we retrieve gets pasted into the prompt, so the pieces in the index need to be the right size. Check what Part A's document index finds for Sara's second question. The answer is in the user guide's "Factory reset" paragraph.

In [10]:
print("One vector per document:"); show(search(QUESTIONS[1], doc_index))

guide = dict(DOCS)["AeroBeat Pro 2 user guide"]
count = lambda text: len(embedder.tokenizer(text)["input_ids"])
print(f"\nMiniLM reads at most {embedder.max_seq_length} tokens. User guide: {count(guide)} tokens. "
      f"Reset section starts at token {count(guide.split('Factory reset')[0])}.")

def chunk(title, text):                               # one paragraph per chunk
    return [(title, p.strip()) for p in text.split("\n\n") if p.strip()]

chunks = [c for title, text in DOCS for c in chunk(title, text)]
print(chunks)
index = build_index(chunks)
print(f"\nOne vector per chunk ({len(DOCS)} documents -> {len(chunks)} chunks, "
      f"longest {max(count(t) for t in index['texts'])} tokens):")
for score, title, text in search(QUESTIONS[1], index):
    print(f"  {score:6.3f}  {title}: {text[len(title) + 2:][:55]}...")

One vector per document:
   0.393  AeroBeat Pro 2 user guide
   0.391  Cleaning and ear cushions
   0.368  AeroBeat app

MiniLM reads at most 256 tokens. User guide: 440 tokens. Reset section starts at token 356.
[('Returns and refunds', 'You can return any AeroBeat product within 45 days of delivery for a full refund. Start a return from the Orders page and print the prepaid label. Products must include the original box and all accessories. Refunds go to your original payment method within 7 business days of the item reaching our warehouse.'), ('Shipping and delivery', 'We send every order with tracking. Standard shipping takes 3 to 5 business days and is free on orders over $50. If you would like faster delivery, Express shipping takes 1 to 2 business days and costs $12.'), ('Warranty', 'Every AeroBeat product comes with a 2-year limited warranty that covers manufacturing defects, such as a dead speaker or a cracked hinge. We repair or replace the product at no cost. The warranty doe

With one vector per document, the user guide still comes first, but only just (0.393 against 0.391 for Cleaning and ear cushions). It matched on general headphone words, not on the reset instructions. After chunking, the "Factory reset" paragraph wins clearly with 0.651. Notice the warning printed while counting tokens: `440 > 256`. That is the tokenizer telling us the guide is longer than MiniLM can read.

The user guide causes three problems as a single vector:

1. **Dilution.** One vector must summarize nine topics (fit, controls, multipoint, reset, ...), so it is not strongly about any of them.
2. **Truncation, and it is silent.** MiniLM reads only its first 256 tokens and drops the rest without a warning. The reset section starts well past that point, so its words **never reached the embedding at all**. No similarity score can find text the model never read.
3. **Prompt cost.** Even when the guide is retrieved, all of it gets pasted, burying a one-sentence answer inside hundreds of tokens about fit and multipoint.

**Chunking** fixes all three: each paragraph gets its own vector, every chunk fits in 256 tokens, and only the relevant paragraph is pasted. Our guide has one topic per paragraph, so splitting on blank lines works. For text without clear paragraphs, a common choice is overlapping windows of a few sentences, so a fact cut by one boundary still appears whole in the next chunk. Each chunk keeps its document title so the model can cite it.

From here on, RAG searches `index`, the chunk index.

## RAG in code

Three prompt choices matter for a 1.5B model:

- The sources go in the **user** message, because small models treat system-message notes as if the user wrote them.
- **Question first, instruction last.** Small models follow the end of the prompt most closely, so the instruction comes right before the answer.
- **A soft way out.** "Say that you don't know" lets the model admit a missing answer without a fixed phrase to hide behind.

In [8]:
def rag_messages(question, hits):                       # step 2: augment
    sources = "\n".join(f"[{i}] {text}" for i, (_, _, text) in enumerate(hits, 1))
    return [{"role": "system", "content": SUPPORT},
            {"role": "user", "content":
                f"Question: {question}\n\nSources:\n{sources}\n\n"
                "Answer the question using the sources above, and cite the source number, like [1]. "
                "If none of the sources answers the question, say that you don't know."}]

def rag_answer(question, k=3, show_prompt=False):
    hits = search(question, index, k)                   # step 1: retrieve
    messages = rag_messages(question, hits)
    if show_prompt:
        print(to_prompt(messages))
    print("Q:", question)
    print("Sources:", [f"{title} ({score:.2f})" for score, title, _ in hits])
    print(f"Prompt tokens: {count_tokens(messages)}")
    print("A:", generate(messages), "\n")               # step 3: generate

rag_answer(QUESTIONS[0], show_prompt=True)
rag_answer(QUESTIONS[1])

<|im_start|>system
You are a customer support assistant for AeroBeat, an online headphone shop.<|im_end|>
<|im_start|>user
Question: How many days do I have to return my AeroBeat headphones?

Sources:
[1] Returns and refunds. You can return any AeroBeat product within 45 days of delivery for a full refund. Start a return from the Orders page and print the prepaid label. Products must include the original box and all accessories. Refunds go to your original payment method within 7 business days of the item reaching our warehouse.
[2] AeroBeat Pro 2 user guide. Sleep mode. If the headphones are not connected for 30 minutes, they turn off to save battery. Fold the earcups flat to put them into deep sleep for travel.
[3] AeroBeat app. The free AeroBeat app for iOS and Android lets you adjust the equalizer, choose noise cancelling levels, and install firmware updates. Turn on Find My Headphones in the app to see where your headphones were last connected.

Answer the question using the sourc

Read the printed prompt first: it is everything the model sees. RAG adds no new model machinery; it is a better-built prompt.

Both answers now carry AeroBeat's real details (45 days; both volume buttons for 15 seconds) with a citation you can check against the listed source. The reset answer may also drift into the Sleep mode chunk. That text is copied from source [2], not invented, but it is noise, and one more reason to paste only a few chunks.

**Prompt wording matters here (Week 7).** An earlier version of this prompt put the instruction first and ended it with "If the sources do not contain the answer, reply exactly: I don't know." With the same three sources, the model answered "I don't know" to the return question, although source [1] says 45 days. Removing the way out entirely made it worse in the other direction: it answered "45 days ... within 48 hours", inventing the second number. The second question only works because of B.3: without chunking, the reset paragraph was never embedded, so no search could have found it.

The price is prompt length: three chunks add a few hundred tokens to every question. That is why we paste the top 3 and not the whole help center. It is also why we paste 3 and not 1: as A.4 showed, the right chunk is not always first, but it is usually in the top 3.

Search always returns its top $k$, even when nothing is relevant. Watch the source scores and the answer.

In [9]:
new_q = "Do you offer a student discount?"
rag_answer(new_q)

# AeroBeat launches a student discount. Add one article and rebuild the index. No training.
DOCS.append(("Student discount", "Students get 15% off any AeroBeat product. Verify your student status once with your school email address at checkout."))
chunks = [c for title, text in DOCS for c in chunk(title, text)]
index = build_index(chunks)
rag_answer(new_q)

Q: Do you offer a student discount?
Sources: ['Payment methods (0.32)', 'Price match (0.16)', 'AeroBeat Buds charging case (0.10)']
Prompt tokens: 206
A: I don't know. None of the provided sources mention a student discount. 

Q: Do you offer a student discount?
Sources: ['Student discount (0.58)', 'Payment methods (0.32)', 'Price match (0.16)']
Prompt tokens: 181
A: Yes, AeroBeat offers a student discount. Students can get 15% off any AeroBeat product by verifying their student status with their school email address during checkout. [1] 



**First call.** The three sources are simply the closest chunks, and none of them mentions a discount. Compare their scores with the ones in B.4. The model replies that it doesn't know, because none of the sources mentions a discount: the instruction worked. Without that instruction, the same model answered "No, AeroBeat does not offer a student discount", a confident claim that no source supports. Keeping a small model grounded through prompting alone is fragile, and Week 12 attacks it by fine-tuning the model to answer only from the given context.

**Second call.** One new article, re-embedded in about a second, and the model answers correctly. Updating a RAG system's knowledge means editing documents, not retraining weights. This is the main reason RAG is so widely used.

**Try it.** Add your own article to `DOCS` (a new product, a holiday return policy) and ask about it.

## Where RAG still fails

| Failure mode | What happens |
|---|---|
| **Retrieval misses** | The relevant chunk does not appear in the top-\(k\) results, so the model answers using irrelevant or incomplete evidence. |
| **Poor ranking** | Similar relevance scores (e.g., A.4's 0.308 vs. 0.303) can rank the best chunk too low, causing it to be overlooked. |
| **Unsupported claims** | The model adds details not supported by the sources (e.g., "within 48 hours") or asserts facts that no source establishes. |
| **Over-refusal** | The model says "I don't know" even when the answer is present in the retrieved sources. |
| **Prompt injection** | A retrieved document contains malicious instructions (e.g., "ignore your instructions"), which the model may mistakenly follow. |
| **High cost** | Every chunk is included in every prompt, increasing token usage, latency, and inference costs. |